# Minimal-channel DNS at $Re_\tau = 180$ — overnight sessions on an A100

The production run for Section 10 of the paper: velocity–vorticity–pressure FOSLS,
spectral elements, condensed vertex-patch Schwarz preconditioner, 6×18 elements at
$N=8$ with 32 Fourier modes in $z$, continuing run01 from $t = 4.96$.

**One session is not enough and the notebook is built around that.**  Every session
picks up the newest checkpoint, marches until a wall-clock deadline it sets for
itself, and leaves a checkpoint on Drive.  Run the same cells tomorrow night and the
flow continues — the convective history and the running statistics travel with the
state, so a restart is not a restart of the flow.

**Tonight, in order:** cell 1 (GPU), 2 (code), 3 (Drive), 5 (smoke test, 2 min),
**6 (the run — set the hours first)**.  Tomorrow morning: cells 7 and 8.

Keep the browser tab open.  Colab disconnects an idle browser after about 90
minutes unless you have Pro+ background execution; a disconnect costs only the
steps since the last sync, but it does end the session.


In [ ]:
#@title 1. Which GPU, and how much of it
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
import subprocess
name = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],
                      capture_output=True, text=True).stdout.strip()
print('GPU:', name)
if 'A100' not in name:
    print('\n!! Not an A100.  This run is fp64 throughout: an L4 or T4 does fp64 at'
          '\n   1/32-1/64 of its fp32 rate and will be ~20x slower than the numbers below.'
          '\n   Runtime -> Change runtime type -> A100, or stop here.')


In [ ]:
#@title 2. The code (public repo, branch main)
import os
if os.path.isdir('/content/lssem/.git'):
    !cd /content/lssem && git fetch -q origin main && git reset -q --hard origin/main
else:
    !git clone -q --branch main https://github.com/chandc/Python_SEM.git /content/lssem
%cd /content/lssem
!git log --oneline -1
!pip install -q numba scipy matplotlib ninja
import torch, numpy
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| numpy', numpy.__version__)


In [ ]:
#@title 3. Drive: where the run lives between sessions
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/lssem_dns'      #@param {type:"string"}
SEED  = '/content/drive/MyDrive/lssem_data/checkpoint_0006200.npz'  #@param {type:"string"}
OUT   = '/content/run02'
os.makedirs(DRIVE, exist_ok=True)

print('state on Drive:')
!ls -la {DRIVE} 2>/dev/null | tail -12
print('\nstatistics snapshots (these are what the averaging window is built from):')
!python colab/stats_window.py --list {DRIVE} 2>/dev/null | tail -8
print('\nlast lines of the run log:')
!tail -5 {DRIVE}/run.log 2>/dev/null || echo '  (no log yet -- tonight is the first session)'
print('\nseed checkpoint present:', os.path.exists(SEED))


## The settings, and why they are not defaults

| setting | value | why |
|---|---|---|
| weighting | `legacy` | Convection is explicit RKW3, so each implicit stage is a Stokes **projection** with a non-solenoidal right-hand side and the constraint rows must dominate.  The balanced weighting of the 2D work is wrong here: it drives the divergence from $8.6\times10^{-4}$ to $3.3\times10^{-1}$ in one step. |
| preconditioner | `vsbatch`, shared across stages | Condensed vertex-patch Schwarz with a $p=2$ coarse space: 72 CG per stage against Jacobi's 4675, and the step reproduces the Jacobi step in every logged digit.  One build at the middle stage's $c$ serves all three. |
| $\Delta t$ | $8\times10^{-4}$ | run01's value, and **physics-limited rather than stability-limited** — verified in cell 7 below.  Keeping it unchanged is also what lets tonight's samples be merged with run01's. |
| checkpoint | every 100 steps | A disconnect costs at most 100 steps.  Checkpoints are written atomically, so a sync never copies a half-written file. |


In [ ]:
#@title 5. Smoke test (~2 min): does the whole path run on this machine?
#@markdown A failure here is a setup problem, not a performance one.  Skip it on a
#@markdown second or third night, once the path is known to work.
!python -u colab/run_vsbatch_a100.py --quick --out /content/results_quick --backends torch 2>&1 | tail -25


In [ ]:
#@title 6. THE RUN — set the budget, then run this cell and leave it
HOURS    = 10.0   #@param {type:"number"}
TARGET_T = 30.0   #@param {type:"number"}
#@markdown `HOURS` is the wall-clock budget: the run stops itself and syncs before it.
#@markdown Set it to a bit less than you expect the session to last (Colab standard
#@markdown sessions run ~12 h).  `TARGET_T` is where the whole simulation ends, over
#@markdown however many nights that takes — 30 eddy turnovers, of which run01 has 5.
#@markdown
#@markdown The first log line comes after the preconditioner build (~1 min);
#@markdown 30 steps later the cell prints the step rate and what fits in the budget.

!python -u colab/run_channel_dns.py \
    --hours {HOURS} --target-t {TARGET_T} --dt 8e-4 --every 100 \
    --out {OUT} --drive {DRIVE} --seed {SEED} \
    --backend torch --precond vsbatch --weighting legacy --share 1


In [ ]:
#@title 7. Was the step set by physics or by the scheme?
#@markdown Computes the Kolmogorov time from the field itself and compares it with
#@markdown the RKW3 stability limit on the same field.  Run it on tonight's last
#@markdown checkpoint: for a DNS the step must be set by the smaller of the two, and
#@markdown it should be the physical one.
import glob
ck = sorted(glob.glob(f'{OUT}/checkpoint_*.npz')) or sorted(glob.glob(f'{DRIVE}/checkpoint_*.npz'))
print('using', ck[-1])
!python scratch/dns_timescales.py {ck[-1]} --dt 8e-4


In [ ]:
#@title 8. Statistics over a window that excludes the transient
#@markdown `PlaneStats` accumulates running sums, so the average over any window is
#@markdown the difference of two snapshots divided by the difference of their sample
#@markdown counts.  Pick A past the start-up transient — look at the $u_\tau$ history
#@markdown in the log — and B as the newest.  Passing only B averages from $t=0$,
#@markdown which includes run01's transient and is not the production number.
!python colab/stats_window.py --list {DRIVE} | tail -10
A = ''  #@param {type:"string"}
B = ''  #@param {type:"string"}
import glob, os
snaps = sorted(glob.glob(f'{DRIVE}/stats_*.npz'))
A = A or (snaps[0] if snaps else '')
B = B or (snaps[-1] if snaps else '')
if A and B and A != B:
    !python colab/stats_window.py {A} {B} --out {DRIVE}/window.png
    from IPython.display import Image, display
    display(Image(f'{DRIVE}/window.png'))
else:
    print('need two snapshots; there will be more after tonight')


## Tomorrow night

Cells 1, 2, 3, then 6.  Nothing else changes: cell 6 finds tonight's checkpoint on
Drive, copies it locally and continues.  Cell 3 will show how far the run has got and
the last few log lines.

**What to watch in the log.**  `u_tau` should hover near 1 — it is prescribed by the
forcing, so a drift toward 0 means the near-wall cycle has died (the minimal box is
intermittent by design, so this is a finding to report rather than necessarily a
bug, but it must be seen while it happens).  `CFL` must stay under 1.732.  `div`
should stay near $10^{-3}$.  `CG` should stay near 72 per stage; if it climbs, the
preconditioner is stale for the current field.

**Cost.**  At the measured A100 rate a 10-hour session covers roughly 4–15 turnovers
depending on which apply path this build uses, so reaching $t=30$ from run01's
$t=4.96$ takes two to four nights.  The statistics improve as $1/\sqrt{T}$, so
stopping early is a legitimate choice — cell 8 gives the profiles from whatever has
accumulated, and the total-stress balance it prints says whether the average has
converged well enough to quote.
